# Interactive Marker annotation

This notebook is responsible only for selecting the video range, preparing keyframes, and interactively annotating 12 markers. After saving, pass the generated `keyframe_marker_points.json` to `piv_tracking.ipynb` or `optical_flow_tracking.ipynb`.

Coordinates use the original video pixel space: the origin is at the top left, x increases to the right, and y increases downward. Markers manually confirmed as invisible are stored as JSON `null`.


In [17]:
%matplotlib widget
from pathlib import Path
import json
import re
import shutil

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display


In [18]:
# Support launching the notebook from either the project root or the scripts/ directory.
def resolve_project_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / 'scripts').is_dir() and (candidate / 'workspaces').is_dir():
            return candidate
    raise RuntimeError(f'Cannot locate the project root from the current directory: {cwd}')


PROJECT_ROOT = resolve_project_root()

WORKSPACE_ROOT = PROJECT_ROOT / 'workspaces'
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# ===== Users usually only need to edit this section =====
# The input video can be an absolute path or a project-relative path as shown below.
INPUT_VIDEO_PATH=PROJECT_ROOT/'test_data'/'Saggitalplane.mp4'

def make_task_folder_name(video_path):
    """Generate a stable, readable task folder name from the video filename.

    Rules: remove the extension, replace spaces and punctuation with underscores,
    and append _processed. For example, Horses videos - 1.MOV becomes
    Horses_videos_1_processed.
    """
    # stem removes the extension; the regular expression splits the filename on spaces, hyphens, and other punctuation.
    parts=[part for part in re.split(r'[\W_]+',Path(video_path).stem,flags=re.UNICODE) if part]
    # Rejoin the parts with underscores; fall back to video if the filename contains no valid characters.
    base='_'.join(parts) or 'video'
    return f'{base}_processed'


# The same input file maps deterministically to the same task directory, allowing repeated runs to be detected.
TASK_FOLDER_NAME=make_task_folder_name(INPUT_VIDEO_PATH)

# Optional: provide an existing JSON path to skip range selection and review or modify existing annotations.
EXISTING_ANNOTATION_JSON=None

In [9]:
# The marker order determines the manual click order, JSON field order, and output color order.
MARKER_NAMES=[
    'wither','shoulder','elbow','knee','front_fetlock','front_coronet',
    'tuber_coxae','hip','stifle','hock','hind_fetlock','hind_coronet',
]
# Each tuple represents a skeleton edge to draw and use for geometric constraints.
SKELETON_EDGES=[
    ('wither','shoulder'),('shoulder','elbow'),('elbow','knee'),('knee','front_fetlock'),('front_fetlock','front_coronet'),
    ('tuber_coxae','hip'),('hip','stifle'),('stifle','hock'),('hock','hind_fetlock'),('hind_fetlock','hind_coronet'),
]

## 1. Select the video and keyframes

After editing `INPUT_VIDEO_PATH`, select the processing range and 2–10 evenly spaced keyframes. Re-annotation replaces only `input/` and `annotations/`; existing PIV and optical-flow results are preserved, but the corresponding tracking notebook should be rerun after saving new annotations.


In [10]:
# Read video metadata. This does not decode the full video; it reads only the FPS, frame count, and dimensions reported by the container.
def inspect_video(path):
    # VideoCapture opens only the container; it does not load the entire video into memory.
    cap=cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise ValueError(f'Cannot open video: {path}')
    # OpenCV properties may return 0, so use or 0 to handle missing values explicitly before converting to the target type.
    info={
        'fps':float(cap.get(cv2.CAP_PROP_FPS) or 0),
        'frame_count':int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0),
        'width':int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0),
        'height':int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0),
    }
    # Release the file handle immediately after reading the metadata.
    cap.release()
    # A video with no valid FPS, invalid dimensions, or fewer than two frames cannot be used for subsequent interval tracking.
    if info['fps']<=0 or info['frame_count']<2 or info['width']<=0 or info['height']<=0:
        raise ValueError(f'Invalid video metadata: {info}')
    return info

In [11]:
# Seek to and read an arbitrary frame for slider previews and keyframe caching.
def read_video_frame(path,frame_index):
    # Reopen the video for each preview and seek directly to the specified zero-based frame index.
    cap=cv2.VideoCapture(str(path)); cap.set(cv2.CAP_PROP_POS_FRAMES,int(frame_index))
    # read() returns a success flag and a BGR image; close the file immediately after reading.
    ok,frame=cap.read(); cap.release()
    if not ok: raise RuntimeError(f'Could not read frame {frame_index}')
    return frame

In [12]:
# The video range selector only prepares the annotation task; it does not run any tracking algorithm.
class VideoRangeSelector:
    def __init__(self):
        # The video is specified by INPUT_VIDEO_PATH in the configuration section rather than selected from a dropdown.
        self.video_path=Path(INPUT_VIDEO_PATH).expanduser()
        if not self.video_path.is_absolute():
            self.video_path=PROJECT_ROOT/self.video_path
        self.video_path=self.video_path.resolve()
        # The HTML label shows the absolute path currently in use so the input can be verified.
        self.video_label=widgets.HTML(value=f'<b>Input video:</b> {self.video_path}')
        # The main frame slider triggers its callback only after the mouse is released to avoid repeated decoding while dragging.
        self.load_button=widgets.Button(description='Load video info',button_style='primary')
        self.frame=widgets.IntSlider(description='Current frame:',continuous_update=False,layout=widgets.Layout(width='720px'))
        # BoundedIntText restricts input to the valid frame-number range set after loading the video.
        self.start=widgets.BoundedIntText(description='Start frame:',min=0)
        self.end=widgets.BoundedIntText(description='End frame:',min=1)
        self.count=widgets.IntSlider(description='Keyframes:',min=2,max=10,value=5,continuous_update=False)
        self.set_start=widgets.Button(description='Confirm',layout=widgets.Layout(width='72px'))
        self.set_end=widgets.Button(description='Confirm',layout=widgets.Layout(width='72px'))
        self.range_status=widgets.Output(layout=widgets.Layout(width='720px'))
        # confirmed distinguishes default or recently edited values from values explicitly confirmed by the user.
        # _setting_boundary prevents programmatic bulk assignments from being mistaken for manual edits.
        self.start_confirmed=False; self.end_confirmed=False; self._setting_boundary=False
        self.prepare=widgets.Button(description='Prepare keyframes',button_style='success')
        self.confirm_overwrite=widgets.Checkbox(value=False,description='Confirm overwrite of input and annotations',indent=False)
        self.status=widgets.HTML()
        # Disable automatic Matplotlib display and embed the single canvas explicitly to avoid showing duplicate figures.
        with plt.ioff():
            self.preview_fig,self.preview_ax=plt.subplots(figsize=(11,6))
        # selection remains None until preparation succeeds; the next section uses it to decide whether the annotator can start.
        self.selection=None
        # Connect button, slider, and input events to the corresponding methods.
        self.load_button.on_click(self.load)
        self.frame.observe(self.preview,names='value')
        self.set_start.on_click(self.confirm_start)
        self.set_end.on_click(self.confirm_end)
        self.start.observe(self.boundary_edited,names='value')
        self.end.observe(self.boundary_edited,names='value')
        self.prepare.on_click(self.prepare_frames)
        # VBox arranges rows vertically, while HBox places related inputs and buttons side by side.
        display(widgets.VBox([
            widgets.HBox([self.video_label,self.load_button]),self.status,self.frame,
            widgets.HBox([self.start,self.set_start]),
            widgets.HBox([self.end,self.set_end]),
            self.count,self.range_status,
            widgets.HBox([self.prepare,self.confirm_overwrite]),
            self.preview_fig.canvas,
        ]))
        self.update_range_status()
        # Read the full video frame count during initialization so IntSlider does not use its default maximum of 100.
        self.load()

    # Display the current confirmation state directly below the input area.
    def update_range_status(self):
        # Show a specific frame number only for an endpoint whose Confirm button has been clicked.
        start_text=f'Confirmed (frame {self.start.value})' if self.start_confirmed else 'Not confirmed'
        end_text=f'Confirmed (frame {self.end.value})' if self.end_confirmed else 'Not confirmed'
        message=f'Start frame: {start_text}    End frame: {end_text}'
        # Clear the previous status before appending new text so repeated confirmations do not accumulate lines.
        self.range_status.clear_output(wait=True)
        self.range_status.append_stdout(message+'\n')

    def confirm_start(self,_=None):
        # Confirm the value in the input field rather than the currently previewed frame above.
        self.start_confirmed=True; self.update_range_status()

    def confirm_end(self,_=None):
        # Confirm the value in the input field rather than the currently previewed frame above.
        self.end_confirmed=True; self.update_range_status()

    # If a number is edited manually, that endpoint returns to the unconfirmed state and must be confirmed again.
    def boundary_edited(self,change):
        # Do not change the confirmation flags while the program initializes control bounds.
        if self._setting_boundary: return
        # Require reconfirmation of an endpoint after its input field is edited manually.
        if change['owner'] is self.start: self.start_confirmed=False
        if change['owner'] is self.end: self.end_confirmed=False
        self.update_range_status()

    # After loading the video, update all control bounds and set the default range from 0 to the final frame.
    def load(self,_=None):
        # Check the path first instead of exposing an unclear OpenCV error to the user.
        path=self.video_path
        if not path.is_file():
            self.status.value=f'<b style="color:red">Input video does not exist: {path}</b>'
            return
        info=inspect_video(path)
        # Frame numbering starts at 0, so the final frame index is the frame count minus 1.
        last=info['frame_count']-1
        self._setting_boundary=True
        self.frame.max=last; self.frame.value=0
        self.start.max=last; self.start.value=0
        self.end.max=last; self.end.value=last
        self._setting_boundary=False
        # Require the start and end points to be confirmed again after reloading the video.
        self.start_confirmed=False; self.end_confirmed=False; self.update_range_status()
        self.status.value=(f'<b>Total frames:</b> {info["frame_count"]}  <b>FPS:</b> {info["fps"]:.3f}  '
                           f'<b>Resolution:</b> {info["width"]}×{info["height"]}')
        self.preview()

    # Refresh the preview only after the slider is released to avoid stuttering from continuous decoding while dragging.
    def preview(self,*_):
        # Return immediately when the input path is invalid so the slider callback does not continue raising errors.
        if not self.video_path.is_file(): return
        frame=read_video_frame(self.video_path,self.frame.value)
        # OpenCV uses BGR and Matplotlib uses RGB, so swap the color channels before display.
        rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
        # Always redraw the same Axes so the interface contains only one preview image.
        self.preview_ax.clear()
        self.preview_ax.imshow(rgb)
        self.preview_ax.set_title(f'Frame {self.frame.value}')
        self.preview_ax.axis('off')
        self.preview_fig.canvas.draw_idle()

    # Validate the range, sample evenly spaced frames, and write them to the fixed video task directory.
    def prepare_frames(self,_=None):
        # Do not allow unconfirmed default values into the task.
        if not (self.start_confirmed and self.end_confirmed):
            self.status.value='<b style="color:red">Confirm both the start and end frames first.</b>'; return
        start,end,count=int(self.start.value),int(self.end.value),int(self.count.value)
        if start>=end:
            self.status.value='<b style="color:red">The start frame must be less than the end frame.</b>'; return
        # linspace includes both endpoints, and rint rounds the evenly spaced floating-point positions to integer frame indices.
        keyframes=np.rint(np.linspace(start,end,count)).astype(int).tolist()
        # Rounding a very short interval may produce duplicate frames, which must be rejected.
        if len(set(keyframes))!=count:
            self.status.value='<b style="color:red">The range is too short to produce unique keyframes.</b>'; return
        source=self.video_path
        if not source.is_file():
            self.status.value=f'<b style="color:red">Input video does not exist: {source}</b>'; return
        # Generate the task directory name deterministically from the input video filename.
        task_dir=WORKSPACE_ROOT/TASK_FOLDER_NAME
        # Require confirmation before re-annotating the same video; always preserve tracking results.
        if task_dir.exists() and any(task_dir.iterdir()):
            if not self.confirm_overwrite.value:
                self.status.value=(f'<b style="color:#b35c00">Existing results found: {task_dir}. '
                                   'To overwrite the input and annotations, select the confirmation checkbox and click '
                                   '"Prepare keyframes" again. The piv_tracking/ and optical_flow_tracking/ results '
                                   'will be preserved, but may need to be rerun.</b>')
                return
            # Clean only the directories managed by the annotation workflow and preserve both tracking result directories.
            for managed_name in ('input', 'annotations'):
                managed_dir = task_dir / managed_name
                if managed_dir.exists():
                    shutil.rmtree(managed_dir)
        input_dir=task_dir/'input'
        annotation_dir=task_dir/'annotations'
        # input stores the video copy, while annotations stores manual annotations and keyframe previews.
        for directory in (input_dir,annotation_dir):
            directory.mkdir(parents=True,exist_ok=True)
        # copy2 copies the video while preserving metadata such as the original timestamps when possible.
        video_path=input_dir/source.name; shutil.copy2(source,video_path)
        info=inspect_video(video_path)
        # Cache only the small set of keyframes during annotation rather than loading the full video.
        frames={frame:read_video_frame(video_path,frame) for frame in keyframes}
        self.selection={'task_dir':task_dir,'video_path':video_path,'video_info':info,'keyframes':keyframes,'frames':frames}
        self.confirm_overwrite.value=False
        self.status.value=(f'<b style="color:green">Prepared keyframes: {keyframes}<br>'
                           f'Task directory: {task_dir}</b>')




In [13]:
selector=None
annotation_path=None
annotations=None

# Create a new annotation interface by default; if an existing JSON file is provided, restore the task and skip range selection.
if EXISTING_ANNOTATION_JSON is None:
    selector=VideoRangeSelector()
else:
    annotation_path=Path(EXISTING_ANNOTATION_JSON).expanduser()
    if not annotation_path.is_absolute():
        annotation_path=PROJECT_ROOT/annotation_path
    annotation_path=annotation_path.resolve()
    if not annotation_path.is_file():
        raise FileNotFoundError(f'Existing annotation JSON does not exist: {annotation_path}')
    # Recover the task root, video copy, and keyframes from the JSON file.
    existing=json.loads(annotation_path.read_text(encoding='utf-8'))
    task_dir=annotation_path.parent.parent
    video_path=task_dir/existing['video']['stored_path']
    info=inspect_video(video_path)
    keyframes=list(map(int,existing['keyframes']))
    frames={frame:read_video_frame(video_path,frame) for frame in keyframes}
    selector=type('LoadedSelection',(),{})()
    selector.selection={'task_dir':task_dir,'video_path':video_path,'video_info':info,'keyframes':keyframes,'frames':frames}
    # Coordinates are lists in JSON; restore them as tuples for interaction, while null continues to mean manually confirmed as invisible.
    annotations={frame:{name:(None if existing['points'][str(frame)][name] is None else tuple(existing['points'][str(frame)][name])) for name in MARKER_NAMES} for frame in keyframes}
    print('Loaded existing annotations:',annotation_path)

## 2. Click to annotate 12 markers

Before running the next cell, click "Prepare keyframes" above. Circles show clicked coordinates; invisible markers are stored as `null`.

The annotation order is fixed by `MARKER_NAMES`. After each click, the interface advances automatically to the next unprocessed point. Completed points can be selected again from the Marker dropdown and overwritten. Clicks do not record coordinates while the zoom or pan tool is active.

In [14]:
# An empty selection means the video range and keyframe preparation have not been completed.
if selector.selection is None:
    raise RuntimeError('Click "Prepare keyframes" above before running this cell.')

# Store a local reference; the subsequent interface and save function read task paths, frames, and metadata from selection.
selection=selector.selection
if annotations is None:
    annotations={frame:{} for frame in selection['keyframes']}


# Use fixed BGR colors when saving PNG files so each marker has the same color across all keyframes.
ANNOTATION_COLORS=[
    (0,255,255),(0,165,255),(0,0,255),(255,0,255),(255,0,0),(255,255,0),
    (0,255,0),(128,255,0),(255,128,0),(255,0,128),(128,0,255),(0,128,255),
]


def draw_annotated_frame(frame_bgr,frame_points):
    """Draw the skeleton, marker names, and invisible-marker list on a full-resolution frame copy."""
    # Copy the original image so drawing the PNG does not modify the clean keyframe held in memory.
    overlay=frame_bgr.copy()
    # Draw a white line only when both endpoints of a skeleton edge are visible.
    for a,b in SKELETON_EDGES:
        pa,pb=frame_points.get(a),frame_points.get(b)
        if pa is not None and pb is not None:
            cv2.line(overlay,tuple(map(lambda v:int(round(v)),pa)),tuple(map(lambda v:int(round(v)),pb)),(255,255,255),2,cv2.LINE_AA)
    # Collect the names manually marked as invisible and write them together at the bottom of the image.
    invisible=[]
    for index,name in enumerate(MARKER_NAMES):
        point=frame_points.get(name)
        if point is None:
            invisible.append(name); continue
        # OpenCV drawing requires integer pixels; the color is determined by the marker's fixed index.
        x,y=int(round(point[0])),int(round(point[1])); color=ANNOTATION_COLORS[index%len(ANNOTATION_COLORS)]
        cv2.circle(overlay,(x,y),7,color,2,cv2.LINE_AA)
        # Draw thick black text as an outline, then overlay thinner colored text for readability on complex backgrounds.
        cv2.putText(overlay,name,(x+9,max(18,y-7)),cv2.FONT_HERSHEY_SIMPLEX,0.48,(0,0,0),3,cv2.LINE_AA)
        cv2.putText(overlay,name,(x+9,max(18,y-7)),cv2.FONT_HERSHEY_SIMPLEX,0.48,color,1,cv2.LINE_AA)
    if invisible:
        text='Invisible: '+', '.join(invisible)
        cv2.putText(overlay,text,(12,overlay.shape[0]-18),cv2.FONT_HERSHEY_SIMPLEX,0.55,(0,0,0),3,cv2.LINE_AA)
        cv2.putText(overlay,text,(12,overlay.shape[0]-18),cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,255),1,cv2.LINE_AA)
    return overlay


# The annotator maintains only serializable frame → marker → (x, y)/None data.
class MarkerAnnotator:
    def __init__(self,selection,points):
        # points is the single annotation state; history records the value before each edit for undo operations.
        self.selection=selection; self.points=points; self.history=[]
        # The two dropdowns select the current keyframe and the marker to process.
        self.frame_picker=widgets.Dropdown(options=selection['keyframes'],description='Frame:')
        self.marker_picker=widgets.Dropdown(options=MARKER_NAMES,description='Marker:')
        self.progress=widgets.HTML(); self.status=widgets.HTML()
        # Create buttons for frame navigation, undo, invisible, clear, and save operations.
        self.previous=widgets.Button(description='← Previous frame'); self.next=widgets.Button(description='Next frame →')
        self.undo_button=widgets.Button(description='Undo'); self.skip=widgets.Button(description='Invisible')
        self.clear=widgets.Button(description='Clear current frame',button_style='warning')
        self.save_button=widgets.Button(description='Save JSON',button_style='success')
        # Use ioff to disable automatic backend display, then place the same canvas in the VBox so Section 2 always contains a single figure.
        with plt.ioff():
            self.fig,self.ax=plt.subplots(figsize=(12,7))
        # Connect Matplotlib mouse-click events to the coordinate recording function.
        self.fig.canvas.mpl_connect('button_press_event',self.on_click)
        # Bind dropdown and button UI events to their corresponding state-update methods.
        self.frame_picker.observe(self.frame_changed,names='value')
        self.previous.on_click(lambda _:self.move(-1)); self.next.on_click(lambda _:self.move(1))
        self.undo_button.on_click(self.undo); self.skip.on_click(lambda _:self.record(None))
        self.clear.on_click(self.clear_frame); self.save_button.on_click(self.save)
        display(widgets.VBox([widgets.HBox([self.frame_picker,self.marker_picker]),
            widgets.HBox([self.previous,self.next,self.undo_button,self.skip,self.clear,self.save_button]),
            self.progress,self.status,self.fig.canvas]))
        # When the interface opens, select the first unprocessed marker automatically and perform the initial render.
        self.first_missing(); self.render(False)

    @property
    def frame(self): return int(self.frame_picker.value)

    def first_missing(self):
        # A missing field means unprocessed; an existing field with value None means confirmed as invisible.
        missing=[name for name in MARKER_NAMES if name not in self.points[self.frame]]
        if missing: self.marker_picker.value=missing[0]

    # Find the next unprocessed marker across all keyframes; remain at the current position when all are complete.
    def advance(self):
        # Search cyclically starting from the current frame so completing one frame naturally advances to the next keyframe.
        for offset in range(len(self.selection['keyframes'])):
            frame=self.selection['keyframes'][(self.selection['keyframes'].index(self.frame)+offset)%len(self.selection['keyframes'])]
            missing=[name for name in MARKER_NAMES if name not in self.points[frame]]
            if missing:
                self.frame_picker.value=frame; self.marker_picker.value=missing[0]; return

    # Save the previous state before each write so Undo can restore the overwritten coordinate or unannotated state.
    def record(self,value):
        # existed distinguishes restoring an old value from deleting a field that did not previously exist.
        name=self.marker_picker.value; existed=name in self.points[self.frame]
        self.history.append((self.frame,name,existed,self.points[self.frame].get(name)))
        # A value of (x, y) represents a visible coordinate; None means the user clicked Invisible.
        self.points[self.frame][name]=value
        self.advance(); self.render(False)

    # Matplotlib xdata/ydata are already in original image coordinates; clamp them to the image bounds before writing.
    def on_click(self,event):
        # Ignore clicks outside the canvas, events without coordinates, and clicks while the zoom or pan tool is active.
        if event.inaxes is not self.ax or event.xdata is None or event.ydata is None: return
        if getattr(self.fig.canvas.toolbar,'mode',''): return
        # Clamp floating-point coordinates to the original image bounds to avoid saving negative or out-of-range positions.
        h,w=self.selection['frames'][self.frame].shape[:2]
        self.record((float(np.clip(event.xdata,0,w-1)),float(np.clip(event.ydata,0,h-1))))

    def undo(self,_):
        # Do nothing when there is no history.
        if not self.history: return
        # LIFO: restore the frame, marker, and old value from before the most recent operation.
        frame,name,existed,old=self.history.pop()
        if existed: self.points[frame][name]=old
        else: self.points[frame].pop(name,None)
        self.frame_picker.value=frame; self.marker_picker.value=name; self.render(False)

    def clear_frame(self,_):
        # Push each item onto the history stack before clearing so all points can be recovered with successive undo operations.
        for name,value in list(self.points[self.frame].items()): self.history.append((self.frame,name,True,value))
        self.points[self.frame].clear(); self.marker_picker.value=MARKER_NAMES[0]; self.render(False)

    def move(self,delta):
        # Use modulo arithmetic so Previous Frame and Next Frame wrap between the first and last keyframes.
        index=self.selection['keyframes'].index(self.frame)
        self.frame_picker.value=self.selection['keyframes'][(index+delta)%len(self.selection['keyframes'])]

    def frame_changed(self,_): self.first_missing(); self.render(False)

    # Allow saving only after all 12 markers in every frame have been clicked or explicitly marked as invisible.
    # Write to .tmp before replace so an interrupted write cannot leave a partial JSON file.
    def save(self,_=None):
        # Check that all 12 fields exist in every frame; None also counts as processed.
        missing={frame:[name for name in MARKER_NAMES if name not in self.points[frame]] for frame in self.selection['keyframes']}
        missing={frame:names for frame,names in missing.items() if names}
        if missing:
            self.status.value='<b style="color:#b35c00">Incomplete: '+str({k:len(v) for k,v in missing.items()})+'</b>'; return False
        # The later tracking cell reads annotation_path directly, so update the global variable.
        global annotation_path
        task_dir=self.selection['task_dir']; output=task_dir/'annotations'; output.mkdir(parents=True,exist_ok=True)
        info=self.selection['video_info']; video_path=self.selection['video_path']
        # Six-digit zero padding keeps files lexicographically ordered by frame number.
        image_names={frame:f'frame_{frame:06d}_markers.png' for frame in self.selection['keyframes']}
        # The payload records video metadata, the processing range, keyframes, PNG paths, and actual coordinates.
        payload={'schema_version':2,'task_id':task_dir.name,
            'video':{'original_filename':video_path.name,'fps':info['fps'],'frame_count':info['frame_count'],
                     'width':info['width'],'height':info['height'],'stored_path':str(video_path.relative_to(task_dir))},
            'selected_range':{'start_frame':min(self.selection['keyframes']),'end_frame':max(self.selection['keyframes'])},
            'keyframes':self.selection['keyframes'],'marker_names':MARKER_NAMES,
            'annotated_frames':[f'annotations/annotated_frames/{image_names[frame]}' for frame in self.selection['keyframes']],
            'points':{str(frame):{name:self.points[frame][name] for name in MARKER_NAMES} for frame in self.selection['keyframes']}}
        annotation_path=output/'keyframe_marker_points.json'; temporary=output/'keyframe_marker_points.json.tmp'
        # Write JSON to a temporary file first; ensure_ascii=False keeps non-ASCII characters readable, and indent=2 makes manual inspection easier.
        temporary.write_text(json.dumps(payload,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
        # Generate the complete PNG set in a temporary directory first, then replace the old directory only after all files succeed.
        annotated_dir=output/'annotated_frames'; annotated_tmp=output/'annotated_frames.tmp'
        # Remove the temporary image directory left by a previous interrupted run.
        if annotated_tmp.exists(): shutil.rmtree(annotated_tmp)
        annotated_tmp.mkdir(parents=True)
        # Redraw every keyframe from the clean original image so repeated saves do not accumulate old graphics.
        for frame in self.selection['keyframes']:
            overlay=draw_annotated_frame(self.selection['frames'][frame],self.points[frame])
            if not cv2.imwrite(str(annotated_tmp/image_names[frame]),overlay):
                raise RuntimeError(f'Could not save annotated preview PNG for frame {frame}')
        # After all PNG files succeed, replace the old directory as a whole, then replace the final JSON with the temporary JSON.
        if annotated_dir.exists(): shutil.rmtree(annotated_dir)
        annotated_tmp.replace(annotated_dir)
        temporary.replace(annotation_path)
        self.status.value=(f'<b style="color:green">Saved JSON: {annotation_path}<br>'
                           f'Saved {len(image_names)} annotated PNG files: {annotated_dir}</b>'); return True

    def render(self,_preserve=True):
        # Clear the previous view and display the current keyframe after converting it from BGR to RGB.
        self.ax.clear(); rgb=cv2.cvtColor(self.selection['frames'][self.frame],cv2.COLOR_BGR2RGB); self.ax.imshow(rgb)
        # Draw the skeleton first, then points and names so points always appear above the lines.
        points=self.points[self.frame]
        for a,b in SKELETON_EDGES:
            if points.get(a) is not None and points.get(b) is not None:
                self.ax.plot([points[a][0],points[b][0]],[points[a][1],points[b][1]],color='white',linewidth=1)
        for index,name in enumerate(MARKER_NAMES):
            point=points.get(name)
            if point is not None:
                self.ax.scatter(*point,s=45,facecolors='none',edgecolors=f'C{index%10}',linewidths=2)
                self.ax.text(point[0]+6,point[1]-6,name,color='yellow',fontsize=6,bbox={'facecolor':'black','alpha':.2,'pad':1})
        self.ax.set_title(f'Frame {self.frame} — click {self.marker_picker.value}'); self.ax.axis('off')
        # Measure both current-frame progress and total keyframe progress by whether each field exists.
        done=sum(name in points for name in MARKER_NAMES)
        total=sum(name in self.points[f] for f in self.selection['keyframes'] for name in MARKER_NAMES)
        self.progress.value=f'<b>Current frame:</b> {done}/12  <b>Total:</b> {total}/{len(self.selection["keyframes"])*12}'
        self.fig.canvas.draw_idle()



In [16]:
annotator=MarkerAnnotator(selection,annotations)

## 3. Run tracking

After successfully clicking "Save JSON", set the output path in `ANNOTATION_JSON_PATH` in the target tracking notebook. PIV and LK optical flow can run independently without overwriting each other's results.
